In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime 

CATALOG = dbutils.widgets.get("catalog")
RAW_SCHEMA = dbutils.widgets.get("stream_schema")
TITLE = dbutils.widgets.get("title")

BASE_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/raw/{RAW_SCHEMA}"

SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{TITLE}"
SILVER_CHECKPOINT = f"{BASE_PATH}/checkpoints/silver"

In [0]:
%run ./03_silver_cleaning

In [0]:
def merge_scd1(micro_batch_df, batch_id):
    print(f"Processing batch: {batch_id}")

    dedup_window = (
        Window
        .partitionBy("show_id")
        .orderBy(F.col("ingestion_time").desc())
    )

    deduplicated_df = (
        micro_batch_df
        .withColumn("_row_number", F.row_number().over(dedup_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    if not spark.catalog.tableExists(SILVER_TABLE):
        (
            deduplicated_df.limit(0).write
                .format("delta")
                .saveAsTable(SILVER_TABLE)
        )
    
    target = DeltaTable.forName(spark, SILVER_TABLE)
    business_columns = [
        col for col in deduplicated_df.columns
        if col not in ["silver_created_at", "silver_updated_at"]
    ]

    update_map = {
        col: f"source.{col}"
        for col in business_columns
    }

    update_map["silver_updated_at"] = "source.silver_updated_at"

    insert_map = {
        col: f"source.{col}"
        for col in business_columns
    }

    insert_map["silver_created_at"] = "source.silver_created_at"
    insert_map["silver_updated_at"] = "source.silver_updated_at"

    (
        target.alias("target")
        .merge(
            deduplicated_df.alias("source"),
            "target.show_id = source.show_id"
        )
        .withSchemaEvolution()
        .whenMatchedUpdate(set=update_map)
        .whenNotMatchedInsert(values=insert_map)
        .execute()
    )

In [0]:
query = (
    silver_df.writeStream
        .foreachBatch(merge_scd1)
        .option("checkpointLocation", SILVER_CHECKPOINT)
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()